In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
import optuna
import math
import time
import numpy as np
import pandas as pd
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## TCN (Temporal Convolutional Network): 

- Modelo de aprendizado profundo que se concentra no processamento e análise de dados de séries temporais
- Realizado pré-processamento para limpeza de dados, corrigindo os formatos numéricos

In [24]:
dataset_path = 'data_train.csv'
df = pd.read_csv(dataset_path, sep="\t")
y = df['y']
df = df.iloc[:, :500]
df = pd.concat([df, y], axis=1)

In [25]:
def prepare_data(df: pd.DataFrame, seq_len: int = 500, target_col: str | None = "y", 
                 test_size: float = 0.2, val_size: float = 0.1, seed: int = 42):
    # Identify target
    if target_col is None or target_col not in df.columns:
        target_col = df.columns[-1]
    y = df[target_col].astype(float).values.reshape(-1, 1)
    
    # Identify feature columns: first ones with length 'seq_len' if present, otherwise all except target
    feature_cols = [c for c in df.columns if c != target_col]
    if len(feature_cols) >= seq_len:
        X = df[feature_cols[:seq_len]].astype(float).values
    else:
        raise ValueError(f"Expected at least {seq_len} feature columns, got {len(feature_cols)}.")
    
    # Handle NaNs if needed
    if np.isnan(X).any():
        # Simple fill (median per timestep). Adjust if you have a preferred imputation.
        med = np.nanmedian(X, axis=0)
        inds = np.where(np.isnan(X))
        X[inds] = np.take(med, inds[1])
    if np.isnan(y).any():
        y = np.nan_to_num(y, nan=np.nanmedian(y))
    
    # Train/test split first (to avoid leakage)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed
    )
    
    # From the train set, carve out validation
    n_val = int(val_size * len(X_train))
    n_train = len(X_train) - n_val
    # We'll split after converting to tensors (using torch random_split)

    # Scale inputs (standardize each timestep feature over the TRAIN set)
    x_scaler = StandardScaler()
    X_train_scaled = x_scaler.fit_transform(X_train)
    X_test_scaled  = x_scaler.transform(X_test)

    # Scale targets to [0,1] for stable training (inverse later for metrics)
    y_scaler = MinMaxScaler()
    y_train_scaled = y_scaler.fit_transform(y_train)
    y_test_scaled  = y_scaler.transform(y_test)

    # To tensors; TCN expects shape (batch, channels=1, length)
    X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32).unsqueeze(1)  # [N, 1, 500]
    X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32).unsqueeze(1)
    y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32)              # [N, 1]
    y_test_t  = torch.tensor(y_test_scaled,  dtype=torch.float32)
    
    full_train = TensorDataset(X_train_t, y_train_t)
    # Split into train/val
    g = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(full_train, [n_train, n_val], generator=g)
    test_ds = TensorDataset(X_test_t, y_test_t)
    
    return train_ds, val_ds, test_ds, x_scaler, y_scaler

In [10]:
train_ds, val_ds, test_ds, x_scaler, y_scaler = prepare_data(df)

In [26]:
class Chomp1d(nn.Module):
    def __init__(self, size):
        super().__init__()
        self.size = size

    def forward(self, x):
        return x[:, :, :-self.size] if self.size > 0 else x


class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout):
        super().__init__()

        self.conv1 = nn.Conv1d(
            n_inputs, n_outputs, kernel_size,
            stride=stride, dilation=dilation, padding=padding
        )
        self.chomp1 = Chomp1d(padding)
        self.relu1  = nn.ReLU()
        self.drop1  = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(
            n_outputs, n_outputs, kernel_size,
            stride=stride, dilation=dilation, padding=padding
        )
        self.chomp2 = Chomp1d(padding)
        self.relu2  = nn.ReLU()
        self.drop2  = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(n_inputs, n_outputs, kernel_size=1) \
            if n_inputs != n_outputs else None

        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.relu1(out)
        out = self.drop1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.relu2(out)
        out = self.drop2(out)

        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)


class TCN(nn.Module):
    def __init__(self, in_channels=1, num_levels=6, hidden_channels=64,
                 kernel_size=5, dropout=0.1, out_dim=1):
        super().__init__()

        layers = []
        channels = in_channels

        for i in range(num_levels):
            dilation = 2 ** i
            padding = (kernel_size - 1) * dilation

            layers.append(
                TemporalBlock(
                    channels, hidden_channels, kernel_size,
                    stride=1, dilation=dilation,
                    padding=padding, dropout=dropout
                )
            )
            channels = hidden_channels

        self.network = nn.Sequential(*layers)

        # Global Average Pooling + Linear
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_channels, out_dim)
        )

    def forward(self, x):
        x = self.network(x)  # [B, C, L]
        x = self.head(x)     # [B, 1]
        return x

In [27]:
class TrainConfig:
    batch_size = 64
    lr = 1e-3
    max_epochs = 150
    patience = 15
    weight_decay = 1e-4
    device = "cuda" if torch.cuda.is_available() else "cpu"

In [28]:
def train_model(train_ds, val_ds, model, cfg=TrainConfig):
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size)

    model = model.to(cfg.device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    best_val = float("inf")
    best_state = None
    no_improve = 0

    for epoch in range(cfg.max_epochs):

        # Train phase
        model.train()
        train_loss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(cfg.device), yb.to(cfg.device)

            optimizer.zero_grad()
            pred = model(xb)
            loss = criterion(pred, yb)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * xb.size(0)

        train_loss /= len(train_loader.dataset)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(cfg.device), yb.to(cfg.device)
                pred = model(xb)
                loss = criterion(pred, yb)
                val_loss += loss.item() * xb.size(0)

        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)

        print(f"Epoch {epoch+1:03d} | train {train_loss:.5f} | val {val_loss:.5f}")

        # Early stopping
        if val_loss < best_val:
            best_val = val_loss
            best_state = model.state_dict()
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= cfg.patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)
    return model

In [29]:
def run_tcn(df, seq_len=500, target_col="y"):
    train_ds, val_ds, test_ds, x_scaler, y_scaler = prepare_data(
        df, seq_len, target_col
    )

    model = TCN(
        in_channels=1,
        num_levels=6,
        hidden_channels=64,
        kernel_size=5,
        dropout=0.1,
        out_dim=1
    )

    model = train_model(train_ds, val_ds, model)

    # Evaluate
    test_loader = DataLoader(test_ds, batch_size=128)
    preds_scaled, ys_scaled = [], []

    model.eval()
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(TrainConfig.device)
            pred = model(xb).cpu().numpy()
            preds_scaled.append(pred)
            ys_scaled.append(yb.numpy())

    preds_scaled = np.vstack(preds_scaled)
    ys_scaled = np.vstack(ys_scaled)

    y_pred = y_scaler.inverse_transform(preds_scaled).ravel()
    y_true = y_scaler.inverse_transform(ys_scaled).ravel()

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)

    print("\n=== Test Metrics ===")
    print(f"MAE : {mae:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"R²  : {r2:.4f}")

    return model, y_true, y_pred

In [30]:
model, y_true, y_pred = run_tcn(df)

Epoch 001 | train 0.12594 | val 0.02551
Epoch 002 | train 0.02724 | val 0.02046
Epoch 003 | train 0.02369 | val 0.01886
Epoch 004 | train 0.02215 | val 0.01791
Epoch 005 | train 0.02198 | val 0.01848
Epoch 006 | train 0.02189 | val 0.01973
Epoch 007 | train 0.02247 | val 0.01790
Epoch 008 | train 0.02147 | val 0.01816
Epoch 009 | train 0.02160 | val 0.01856
Epoch 010 | train 0.02112 | val 0.01725
Epoch 011 | train 0.02213 | val 0.01717
Epoch 012 | train 0.02114 | val 0.01692
Epoch 013 | train 0.02025 | val 0.01665
Epoch 014 | train 0.01964 | val 0.01587
Epoch 015 | train 0.02020 | val 0.01752
Epoch 016 | train 0.01887 | val 0.01929
Epoch 017 | train 0.02019 | val 0.01461
Epoch 018 | train 0.01857 | val 0.01415
Epoch 019 | train 0.01724 | val 0.01715
Epoch 020 | train 0.01628 | val 0.01404
Epoch 021 | train 0.01512 | val 0.01404
Epoch 022 | train 0.01421 | val 0.01334
Epoch 023 | train 0.01407 | val 0.01331
Epoch 024 | train 0.01408 | val 0.02491
Epoch 025 | train 0.01659 | val 0.01211
